In [1]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning, message=".*escape sequence.*")

from textwrap import dedent
import httpx
import json
import requests
import pandas as pd
import numpy as np
import re
import requests
import uuid
from openai import OpenAI
import time
import os
from dotenv import load_dotenv, find_dotenv

from pydantic import BaseModel, Field
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser, BaseOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


load_dotenv(find_dotenv(usecwd=True))
use = 'olm'

if use == 'hf':
    os.environ["BASE_URL"] = "https://router.huggingface.co/v1"
    os.environ["MODEL_NAME"] ='Qwen/Qwen3-32B'
    os.environ["OPENAI_API_KEY"] = os.getenv("HUGGINGFACE_HUB_TOKEN2")
    
elif use == 'olm':
    os.environ["BASE_URL"] = "http://localhost:11434/v1"
    os.environ["MODEL_NAME"] ='qwen3:14b'
    os.environ["OPENAI_API_KEY"] = 'hey'
    
else:
    os.environ["BASE_URL"] = "https://gigachat.devices.sberbank.ru/api/v1"
    os.environ["MODEL_NAME"] ='GigaChat-Max'
    
    url = "https://ngw.devices.sberbank.ru:9443/api/v2/oauth"
    # Создадим идентификатор UUID (36 знаков)
    rq_uid = str(uuid.uuid4())
    payload={
      'scope': 'GIGACHAT_API_PERS'
    }
    headers = {
      'Content-Type': 'application/x-www-form-urlencoded',
      'Accept': 'application/json',
      'RqUID': rq_uid,
      'Authorization': f'Basic {os.getenv("GIGACHAT_AUTH")}'
    }

    response = requests.request("POST", url, headers=headers, data=payload, verify=False)
    os.environ["OPENAI_API_KEY"] = response.json()['access_token']
    

os.environ['TEMPERATURE'] = '0'


# for importing trialmind, api_key should be set beforehand
from trialmind.pubmed import pmid2papers, PubmedAPIWrapper, pmid2biocxml, parse_bioc_xml, pmid2fulltext
from trialmind.api import StudyCharacteristicsExtraction, ScreeningCriteriaGeneration,\
                            LiteratureScreening, ScreeningCriteriaCTGeneration,\
                            CTScreening
from trialmind.retrievers import split_text_into_chunks
import extract
import time
from markdown_pdf import MarkdownPdf
from markdown_pdf import Section
import ast
import report_gen
%load_ext autoreload
%autoreload 2

## preloading; checking

In [3]:
# preloading the model to speed up calcs
for i in ['qwen3:0.6b','qwen3:8b','qwen3:14b']:
    os.environ["MODEL_NAME"] =i

    client = OpenAI(
            base_url=os.getenv("BASE_URL"),
            api_key=os.getenv("OPENAI_API_KEY"),
            http_client=httpx.Client(verify=False)
        )
    response = client.chat.completions.create(
            model=os.getenv("MODEL_NAME"),
            messages=[{'role':'user','content':''}],  # Ensure messages is a list
            temperature=0,
        extra_body={"reasoning_effort": "none"}
        )

os.environ["MODEL_NAME"] = 'qwen3:0.6b'
start = time.time()
response = client.chat.completions.create(
        model=os.getenv("MODEL_NAME"),
        messages=[{'role':'user','content':'hey'}],  # Ensure messages is a list
        temperature=0,
    extra_body={"reasoning_effort": "none"}
    )
end = time.time()
print(end - start)

response#.choices[0]#.message.content

2.2828001976013184


ChatCompletion(id='chatcmpl-366', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you today? Let me know what you need! 😊', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1773839441, model='qwen3:0.6b', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=19, prompt_tokens=17, total_tokens=36, completion_tokens_details=None, prompt_tokens_details=None))

In [2]:
openai_client = OpenAI(
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    http_client=httpx.Client(verify=False)
)
response = openai_client.chat.completions.create(
        model='qwen3:8b',
        messages=[{'role':'user','content':'hey'}],
        temperature=0,
        extra_body={"reasoning_effort": "none"}
    )
response

ChatCompletion(id='chatcmpl-151', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you today? 😊', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1774016961, model='qwen3:8b', object='chat.completion', service_tier=None, system_fingerprint='fp_ollama', usage=CompletionUsage(completion_tokens=12, prompt_tokens=17, total_tokens=29, completion_tokens_details=None, prompt_tokens_details=None))

In [52]:
embeddings = OllamaEmbeddings(model="all-minilm")
vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    collection_metadata={"hnsw:space": "cosine"},
    #persist_directory="./chroma_langchain_db",  
)

text_splitter=RecursiveCharacterTextSplitter(chunk_size=300, 
                                             chunk_overlap=100)

## generating a report

In [16]:
treatements_eng, fin_condition = report_gen.get_res(file_path = "docs/LuC_213_L00_DL_edited_oncobox_ru.pdf",
            model_translate='qwen3:8b', model_main='qwen3:8b',
            n_papers=5,ct_pages=2,
            ct_criteria = \
            ["Does the trial focus on patients with '{fin_condition}'?",
             "Does the trial examine the use or sensitivity of '{treatement}' among main treatments?"],
            papers_criteria=\
            ["Does the study focus on patients/models/cells with '{fin_condition}'?",
             "Does the study examine the use/effect/sensitivity of '{treatement}' among main treatments?", 
             "Does the study describe the effect of '{treatement}' treatment?"],
            ct_screen_thresh=0,
            paper_screen_thresh = [1,0,0],
            save_files=True
           )

INFO:root:qwen3:8b
INFO:root:GETTING INFO FROM FILE
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:7.316885709762573
INFO:root:GETTING CLINICAL TRIALS
INFO:root:(4, 10)
INFO:root:0.3986694812774658
INFO:root:SCREENING CLINICAL TRIALS
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"



parsed_results in asynch [PaperEvaluation(evaluations=['YES', 'YES'], rationale=['The trial explicitly mentions patients with Non-small cell lung cancer (NSCLC).', 'The trial focuses on the use of Afatinib in patients with EGFR driver mutations, which is a main treatment for this population.'])]


INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"



parsed_results in asynch [PaperEvaluation(evaluations=['YES', 'YES'], rationale=['The trial explicitly mentions patients with Non-small cell lung cancer (NSCLC) in both the Phase I and Phase II steps.', 'The trial examines the use of Afatinib (BIBW 2992) as a monotherapy in patients with NSCLC who have failed first-generation EGFR-TKI treatments.'])]


INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"



parsed_results in asynch [Results(fieldresult=[FieldResult(name='Afatinib effectiveness', value='Afatinib showed efficacy in older adult EGFR-mutant lung cancer patients.', source_id=[0, 1, 2])])]


INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:5.605045318603516
INFO:root:COMBINING RESULTS FROM CLINICAL TRIALS



parsed_results in asynch [Results(fieldresult=[FieldResult(name='Afatinib effectiveness', value='Afatinib is approved for first-line treatment in advanced EGFR mutation-positive NSCLC.', source_id=[6])])]
10 [4, 5]
10 [0, 1, 2]
10 [6]


INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:3.40718150138855
INFO:root:Afatinib showed partial reversal of immunosuppressive-like transcriptional signature in resistant NSCLC [[41672191]]. Afatinib showed efficacy in older adult EGFR-mutant lung cancer patients [[41673455]]. Afatinib is approved for first-line treatment in advanced EGFR mutation-positive NSCLC [[41736995]].
--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\user\anaconda3\envs\olm2\Lib\logging\__init__.py", line 1110, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\anaconda3\envs\olm2\Lib\logging\__init__.py", line 953, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\anaconda3\envs\olm2\Lib\logging\__init__.py", line 687, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\user\anaconda3\envs\olm2\Li

In [28]:
report_gen.fill_pdf(treatements_eng, fin_condition, model_translate='qwen3:8b')

INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:23.611688137054443
